In [2]:
using DualNumbers
using ForwardDiff
using LinearAlgebra
using Quadmath

In [6]:
toRadians::Float128 = pi/180
numberOfAtoms::Int64 = 6
masses::Vector{Float128} = Float128.([12.00000000, 15.99491463, 1.00782503223, 1.00782503223, 1.00782503223, 1.00782503223])
valenceCoordinates = Float128.([1.42167426, 0.95631445, 1.09061209, 1.09061209, 1.09061209, 108.10210467*toRadians, 110.28974516*toRadians, 110.28974516*toRadians, 110.28974516*toRadians, 0.0, 0.0, pi/3])

12-element Vector{Float128}:
 1.42167426000000007846324479032773525e+00
 9.56314450000000038087932807684410363e-01
 1.09061209000000003399577508389484137e+00
 1.09061209000000003399577508389484137e+00
 1.09061209000000003399577508389484137e+00
 1.88673765482703823842586014038349470e+00
 1.92491918422748019513219346619590753e+00
 1.92491918422748019513219346619590753e+00
 1.92491918422748019513219346619590753e+00
 0.00000000000000000000000000000000000e+00
 0.00000000000000000000000000000000000e+00
 1.04719755119659763131778618117095903e+00

In [8]:
function defineMolecularFrameXYZ(valence)
    # Stores type to prevent type conversion problems with duals
    T = eltype(valence) 
    molecularFrameXYZ = zeros(T, numberOfAtoms, 3)
    # C at origin
    # CO defines z-axis
    molecularFrameXYZ[2, 3] = valence[1]
    # OH defines x-axis
    molecularFrameXYZ[3, 1] = valence[2]*sin(valence[6])
    molecularFrameXYZ[3, 3] = valence[1]-valence[2]*cos(valence[6])
    # CH1
    molecularFrameXYZ[4, 1] = valence[3]*sin(valence[7])*cos(valence[12] - sqrt(2)*valence[11]/3)
    molecularFrameXYZ[4, 2] = valence[3]*sin(valence[7])*sin(valence[12] - sqrt(2)*valence[11]/3)
    molecularFrameXYZ[4, 3] = valence[3]*cos(valence[7])
    # CH2
    molecularFrameXYZ[5, 1] = valence[4]*sin(valence[8])*cos(4*pi/3 + valence[12] + valence[10]/sqrt(6) + sqrt(2)*valence[11]/6)
    molecularFrameXYZ[5, 2] = valence[4]*sin(valence[8])*sin(4*pi/3 + valence[12] + valence[10]/sqrt(6) + sqrt(2)*valence[11]/6)
    molecularFrameXYZ[5, 3] = valence[4]*cos(valence[8])
    # CH3
    molecularFrameXYZ[6, 1] = valence[5]*sin(valence[9])*cos(2*pi/3 + valence[12] - valence[10]/sqrt(6) + sqrt(2)*valence[11]/6)
    molecularFrameXYZ[6, 2] = valence[5]*sin(valence[9])*sin(2*pi/3 + valence[12] - valence[10]/sqrt(6) + sqrt(2)*valence[11]/6)
    molecularFrameXYZ[6, 3] = valence[5]*cos(valence[9])
    
    return molecularFrameXYZ
end

function transformToCOM(molecularFrame)
    COM = masses'*molecularFrame/sum(masses)
    frameCOM = molecularFrame .- COM
    return frameCOM
end

function defineCOMframe(valence)
    molecularFrame = defineMolecularFrameXYZ(valence)
    frameCOM = transformToCOM(molecularFrame)
    return frameCOM
end

function computeTMatrix(valence)
    frameCOM = defineCOMframe(valence)
    T = eltype(valence)
    tMatrix = zeros(T, numberOfAtoms*3, numberOfAtoms*3)
    for i in 1:6
        # Translational part
        tMatrix[i*3-2:i*3, 1:3] = Matrix(1I, 3, 3)
    
        # Rotational Part
        for j in 1:3
            unitVector = Matrix(1I, 3, 3)[j, :]
            tMatrix[i*3-2:i*3, 3 + j] = cross(unitVector, frameCOM[i, :])
        end
        # Vibrational part
        for j in 1:3
            cartesianValenceGradient = ForwardDiff.gradient(valence -> defineCOMframe(valence)[i, j], valence) 
            tMatrix[i*3-3+j, 7:end] = cartesianValenceGradient
        end
    end
    return tMatrix
end

# Redundant for these purposes - mainly useful for symbolic stuff
function computeSMatrix(valence)
    tMatrix = computeTMatrix(valence)
    sMatrix = inv(tMatrix)
    return sMatrix
end

# Note that the upper case GMatrix is the controvariant metric tensor! gMatrix is the covariante
function computegMatrix(valence)
    tMatrix = computeTMatrix(valence)
    T = eltype(valence)
    gMatrix = zeros(T, 3*numberOfAtoms, 3*numberOfAtoms)
    for i in 1:3*numberOfAtoms
        for j in 1:3*numberOfAtoms
            for n in 1:numberOfAtoms
                gMatrix[i, j] += dot(tMatrix[n*3-2:n*3, i], tMatrix[n*3-2:n*3, j])*masses[n]
            end          
        end
    end
    return gMatrix
end

function computeGMatrix(valence)
    gMatrix = computegMatrix(valence)
    GMatrix = inv(gMatrix)
    return GMatrix
end

computeGMatrix (generic function with 1 method)

In [14]:
@time GMatrix = computeGMatrix(valenceCoordinates)

  0.002991 seconds (4.59 k allocations: 795.844 KiB)


18×18 Matrix{Float128}:
  3.12244206044199514384601315528676773e-02  …   3.45404749955127422106058539909948149e-37
  1.93089197652810859440433838922353891e-71      8.89370346035762478848911454441105352e-36
 -1.81378459403891875229779497269700194e-72      1.70872501214283318410525128186610045e-37
 -4.14146284065376833650701130613113040e-38     -7.19692018272743900944682360186117570e-02
  8.08719311196023789097624230216104854e-37     -3.85219954066737942278153742183623575e-33
  1.12591281645662776448268546883796034e-38  …  -1.31590385924362549633195657240403005e+00
  7.26479926451260514763540885476025329e-38     -5.66460392969546121105844045704321527e-37
 -1.45698033141983784233664422797969352e-36      7.07424239436081298082474405677221945e-34
  6.53082984489074307907573227419748740e-38      1.55644027150270938109045953026931615e-02
 -8.12826309335454286961554213559496167e-37     -1.55644027150271000025026995619735126e-02
  2.39949573052880641276372008877458766e-36  …   6.191598104259282

In [15]:
GMatrix[18, 18]

1.63198069852602447241310294825574524e+00

In [16]:
GMatrix[18, 17]

-4.80892875767042428393724713562899731e-02

In [17]:
GMatrix[4, 4]

7.21631984797708460639297779067141536e-02

In [18]:
GMatrix[5, 5]

7.21631984797708460639297779067141416e-02

In [19]:
GMatrix[7, 7]

1.45853204375620977274330193446118750e-01

In [22]:
1/sum(masses)

3.12244206044199514384601315528676773e-02

In [21]:
GMatrix[1, 1]

3.12244206044199514384601315528676773e-02

In [161]:
GMatrix[2, 2]

3.12244206044199514384601315528676894e-02

In [30]:
GMatrix[3, 3]

3.12244206044199514384601315528676533e-02

In [53]:
Uterm1 = Float128(0.0)
for i in 1:numberOfAtoms
    for j in 1:3
        for k in 1:3
            Uterm1 += sMatrix[3 + j, 3*i - 3 + k]*(sMatrix[3 + j, 3*i - 3 + k] - sMatrix[3 + k, 3*i - 3 + j])/(8*masses[i])
        end
    end
end
Uterm1

2.00569581645338610824963188998765609e-01

In [ ]:
Uterm2 = Float128(0.0)
for i in 1:numberOfAtoms
    for j in 1:3
        for k in 1:3
            Uterm2 += sMatrix[3 + j, 3*i - 3 + k]*(sMatrix[3 + j, 3*i - 3 + k] - sMatrix[3 + k, 3*i - 3 + j])/(8*masses[i])
        end
    end
end


In [94]:
Dual(Float128(0.0), Float128(1.0))

0.00000000000000000000000000000000000e+00 + 1.00000000000000000000000000000000000e+00ɛ

In [79]:
typeof(Dual(2.0, 1.0))

Dual128 (alias for Dual{Float64})

In [75]:
dualpart(tan(Dual(2, 1)))

5.774399204041917

In [167]:
ForwardDiff.derivative(x -> ForwardDiff.derivative(x -> x^2, x), Float128(2.0))

2.00000000000000000000000000000000000e+00

In [169]:
ForwardDiff.derivative(x -> ForwardDiff.derivative(x -> ForwardDiff.derivative(x -> x^3, x), x), Float128(2.0))

6.00000000000000000000000000000000000e+00